# GIADA52i — solution retention
Same52h selected warmstart for four paired arms: constant/decaying step × fit-only current step acceptance. Original52d development guards,10states and thresholds unchanged. Cap52j.

In [ ]:
import hashlib,json,os,shutil,subprocess,sys,time,threading,zipfile,traceback
from pathlib import Path
REVISION='dc7b1ce10a15902b19e1536a98066fbe7d23bae8'
CHECKS=[('source41n.zip.bin', '7e07984c6c8770fee223a83f4694922c2f8cb92529e3df899bc511cf8d5080be'), ('checkpoint45c.zip.bin', '555133b7fc71719ecf459e2b04ffed8fd22a50af198bdb409735e0d8ca996afe'), ('evaluation45d.zip.bin', 'daabec1f2e998bdc538c2cd7303470d895ad30bedf2124159b1bcad6e2ebc2f7'), ('models52e.zip.bin', '8df0d9ec12f5690e20dfee766eedbdf914a2ead984f012280302e69f1692fe49'), ('warmstart52h.zip.bin', 'b7967b717ce71bf2450ad99d0ae4eb2837f53b06bfcc85f2784d8743dfcf7507')]
work=Path('/tmp/giada_task52i');work.mkdir(exist_ok=False)
repo=work/'giada';out=work/'result';public=Path('/kaggle/working');log=public/'process.log';started=time.time();returncode=None
def sha(p):
 with p.open('rb') as f:return hashlib.file_digest(f,'sha256').hexdigest()
def setup(cmd):
 with log.open('a') as stream:subprocess.run(cmd,stdout=stream,stderr=subprocess.STDOUT,check=True)
lock=threading.Lock();stop=threading.Event()
def snapshot(final=False):
 with lock:
  temporary=public/'snapshot_build.zip';name='task52i_final.zip' if final else 'task52i_checkpoint_latest.zip'
  with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_DEFLATED) as z:
   if out.exists():
    for p in out.rglob('*'):
     if p.is_file() and not p.name.endswith('.tmp'):z.write(p,p.relative_to(out))
   if log.exists():z.write(log,'console_process.log')
  temporary.replace(public/name)
  receipt=dict(valid=True,archive=name,sha256=sha(public/name),bytes=(public/name).stat().st_size,revision=REVISION,at=time.time(),worker_exit_code=returncode,thresholds_changed=False)
  rp=public/('kaggle_receipt.json' if final else 'task52i_checkpoint_receipt.json');tmp=rp.with_suffix('.tmp');tmp.write_text(json.dumps(receipt));tmp.replace(rp)
def periodic():
 while not stop.wait(180):
  try:snapshot()
  except Exception as e:print('Backup retry: '+type(e).__name__,flush=True)
try:
 paths=[]
 for name,digest in CHECKS:
  found=list(Path('/kaggle/input').rglob(name));assert len(found)==1,(name,len(found));assert sha(found[0])==digest;paths.append(found[0])
 print('[GIADA52i] all five input hashes verified',flush=True)
 setup([sys.executable,'-m','pip','install','--quiet','torch==2.11.0+cu130','--index-url','https://download.pytorch.org/whl/cu130'])
 for cmd in (['git','clone','--quiet','--depth','1','--branch','codex/surrogate-validity-audit','https://github.com/Zagred47/giada.git',str(repo)],['git','-C',str(repo),'fetch','--quiet','--depth','1','origin',REVISION],['git','-C',str(repo),'checkout','--quiet','--detach','FETCH_HEAD']):setup(cmd)
 env=dict(os.environ,CUBLAS_WORKSPACE_CONFIG=':4096:8',MPLBACKEND='Agg')
 thread=threading.Thread(target=periodic,daemon=True);thread.start()
 with log.open('a') as stream:
  proc=subprocess.Popen([sys.executable,'-u',str(repo/'scripts/run_task52i.py'),*map(str,paths),str(out)],cwd=repo,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
  for line in proc.stdout:
   stream.write(line);stream.flush()
   if line.startswith('[GIADA') or 'Error' in line or 'Traceback' in line:print(line.rstrip(),flush=True)
  returncode=proc.wait()
finally:
 stop.set()
 out.mkdir(exist_ok=True)
 (out/'kaggle_execution.json').write_text(json.dumps(dict(revision=REVISION,resumed_from=0,returncode=returncode,thresholds_changed=False,elapsed_seconds=time.time()-started)))
 snapshot(final=True)
assert returncode==0,'Worker failed: inspect saved logs and checkpoint, do not retrain from zero'
